# Notebook 5B: Tool Guardrails - Protecting Every Tool Call

**What you'll learn:** Why tool guardrails exist, `@tool_input_guardrail`, `@tool_output_guardrail`, `ToolGuardrailFunctionOutput`, real-world scenarios where agent-level guardrails aren't enough.

---

## The Problem: Agent Guardrails Have Blind Spots

Recall from Notebook 5:
- **Input guardrails** run only for the **first** agent in a chain
- **Output guardrails** run only for the **last** agent

But what about **tools**? An agent can call tools dozens of times during a single run. Each tool call can have dangerous arguments or leak sensitive data in its return value.

```
User: "Search for my health records"
        │
        ▼
  Agent (input guardrail passes — message looks fine)
        │
        ├── calls search_database(query="patient SSN 123-45-6789")  ← DANGEROUS ARGS!
        │         └── returns "SSN: 123-45-6789, Diagnosis: ..."    ← SENSITIVE OUTPUT!
        │
        └── Agent response (output guardrail checks this — but the tool already executed!)
```

**Tool guardrails solve this.** They wrap individual tools and run:
- **Before** the tool executes (validate/block the arguments)
- **After** the tool executes (validate/redact the return value)

```
Agent decides to call tool
       │
       ▼
  ┌─────────────────────────┐
  │  TOOL INPUT GUARDRAIL   │  ← Check arguments BEFORE tool runs
  │  (block secrets, PII,   │     Can: allow, reject, or raise tripwire
  │   validate ranges)      │
  └────────┬────────────────┘
           ▼
     Tool Executes
           │
           ▼
  ┌─────────────────────────┐
  │  TOOL OUTPUT GUARDRAIL  │  ← Check return value AFTER tool runs
  │  (redact PII, validate  │     Can: allow, replace output, or raise tripwire
  │   response format)      │
  └─────────────────────────┘
```

In [1]:
# uv add openai-agents

In [2]:
from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel, set_tracing_disabled

set_tracing_disabled(True)

def get_ollama_model(name="minimax-m2.5:cloud"):
    return OpenAIChatCompletionsModel(
        model=name,
        openai_client=AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    )

# MODEL = get_ollama_model() # Uncomment for Ollama Model
MODEL = "gpt-5.4-mini"  

---

## The API: 3 Key Pieces

```python
from agents import (
    function_tool,
    tool_input_guardrail,        # Decorator — runs BEFORE tool executes
    tool_output_guardrail,       # Decorator — runs AFTER tool executes
    ToolGuardrailFunctionOutput, # Return type with 3 options:
                                 #   .allow()           → let it through
                                 #   .reject_content()  → block with message
                                 #   (tripwire_triggered=True) → raise exception
)
```

Then attach guardrails to the tool itself:
```python
@function_tool(
    tool_input_guardrails=[my_input_guard],
    tool_output_guardrails=[my_output_guard],
)
def my_tool(x: str) -> str: ...
```

**Key difference from agent guardrails:** Tool guardrails run on **every invocation** of that tool, even in multi-agent chains with handoffs.

---

## Example 1: Block Secrets in Tool Arguments

**Scenario:** You have a tool that sends data to an external API. The LLM might accidentally include API keys, tokens, or passwords in the arguments.

In [3]:
import json
from agents import (
    Agent, Runner,
    function_tool, tool_input_guardrail,
    ToolGuardrailFunctionOutput,
)


# --- Tool Input Guardrail: Block secrets ---
@tool_input_guardrail
def block_secrets(data):
    """Prevent API keys or tokens from being passed to any tool."""
    args_str = data.context.tool_arguments or "{}"
    
    secret_patterns = ["sk-", "api_key", "password", "secret", "token=", "Bearer "]
    
    for pattern in secret_patterns:
        if pattern.lower() in args_str.lower():
            print(f"TOOL INPUT BLOCKED: Found '{pattern}' in arguments")
            return ToolGuardrailFunctionOutput.reject_content(
                f"Blocked: Remove sensitive data ('{pattern}') before calling this tool."
            )
    
    print("TOOL INPUT OK: No secrets found")
    return ToolGuardrailFunctionOutput.allow()


# --- Tool with guardrail attached ---
@function_tool(
    tool_input_guardrails=[block_secrets],  # <-- Attached here!
)
def send_to_api(endpoint: str, payload: str) -> str:
    """Send data to an external API endpoint."""
    return f"Sent to {endpoint}: {payload[:50]}... (200 OK)"


agent = Agent(
    name="API Agent",
    instructions="You help send data to APIs. Use the send_to_api tool.",
    model=MODEL,
    tools=[send_to_api],
)


# Test 1: Clean call — should pass
print("--- Test 1: Clean API call ---")
result = await Runner.run(agent, 'Send payload "hello world" to endpoint https://api.example.com/data')
print(f"Result: {result.final_output[:150]}")

# Test 2: Secret in arguments — should be blocked
print("\n--- Test 2: Secret in arguments ---")
result = await Runner.run(
    agent,
    'Send to https://api.example.com/auth with payload "api_key=sk-12345secret"'
)
print(f"Result: {result.final_output[:200]}")

--- Test 1: Clean API call ---
TOOL INPUT OK: No secrets found
Result: Done.

--- Test 2: Secret in arguments ---
TOOL INPUT BLOCKED: Found 'sk-' in arguments
Result: I can’t send that payload because it contains a sensitive API key. Please redact the secret first, then I can help send the request.


### What happened:

```
Test 1:  Agent calls send_to_api(endpoint="...", payload="hello world")
         → block_secrets checks args → no secrets → .allow() → tool runs → response

Test 2:  Agent calls send_to_api(endpoint="...", payload="api_key=sk-12345...")
         → block_secrets checks args → finds "sk-" → .reject_content() 
         → tool DOES NOT execute → rejection message sent back to LLM
         → LLM sees the rejection and responds to the user
```

With `.reject_content()`, the tool never runs, but the agent loop continues — the LLM receives the rejection message and can respond accordingly.

---

## Example 2: Redact PII from Tool Output

**Scenario:** A database lookup tool returns full customer records including SSNs and credit card numbers. The output guardrail redacts sensitive data before the LLM sees it.

In [4]:
import re
from agents import (
    Agent, Runner,
    function_tool, tool_output_guardrail,
    ToolGuardrailFunctionOutput,
)


# --- Tool Output Guardrail: Redact PII ---
@tool_output_guardrail
def redact_pii(data):
    """Redact SSNs and credit card numbers from tool output."""
    text = str(data.output or "")
    original = text
    
    # Redact SSN patterns (XXX-XX-XXXX)
    text = re.sub(r'\b(\d{3})-(\d{2})-(\d{4})\b', r'***-**-\3', text)
    
    # Redact credit card patterns (XXXX-XXXX-XXXX-XXXX)
    text = re.sub(r'\b\d{4}[- ]?\d{4}[- ]?\d{4}[- ]?(\d{4})\b', r'****-****-****-\1', text)
    
    # Redact email addresses
    text = re.sub(r'[\w.+-]+@[\w-]+\.[\w.-]+', '[EMAIL REDACTED]', text)
    
    if text != original:
        print("TOOL OUTPUT REDACTED: PII found and masked")
        # Replace the output with redacted version
        return ToolGuardrailFunctionOutput.reject_content(text)
    
    print("TOOL OUTPUT OK: No PII found")
    return ToolGuardrailFunctionOutput.allow()


# --- Tool with output guardrail ---
@function_tool(
    tool_output_guardrails=[redact_pii],
)
def lookup_customer(customer_id: str) -> str:
    """Look up customer details by ID."""
    # Simulated database record with sensitive data
    records = {
        "C001": (
            "Name: Ahmed Khan\n"
            "Email: ahmed.khan@gmail.com\n"
            "SSN: 123-45-6789\n"
            "Card: 4532-1234-5678-9012\n"
            "Plan: Enterprise\n"
            "Balance: PKR 125,430"
        ),
        "C002": (
            "Name: Sara Ali\n"
            "Plan: Pro\n"
            "Balance: PKR 4,900"
        ),
    }
    return records.get(customer_id, f"Customer {customer_id} not found")


agent = Agent(
    name="Support Agent",
    instructions="You look up customer info using the lookup_customer tool. Share the details with the user.",
    model=MODEL,
    tools=[lookup_customer],
)


# Test 1: Customer with PII — output guardrail should redact
print("--- Test 1: Customer with PII ---")
result = await Runner.run(agent, "Look up customer C001")
print(f"Agent sees (after redaction): {result.final_output[:300]}")

# Test 2: Customer without PII — should pass through
print("\n--- Test 2: Customer without PII ---")
result = await Runner.run(agent, "Look up customer C002")
print(f"Agent sees: {result.final_output[:200]}")

--- Test 1: Customer with PII ---
TOOL OUTPUT REDACTED: PII found and masked
Agent sees (after redaction): Customer C001 details:

- Name: Ahmed Khan
- Email: [EMAIL REDACTED]
- SSN: ***-**-6789
- Card: ****-****-****-9012
- Plan: Enterprise
- Balance: PKR 125,430

--- Test 2: Customer without PII ---
TOOL OUTPUT OK: No PII found
Agent sees: Customer C002 details:

- Name: Sara Ali
- Plan: Pro
- Balance: PKR 4,900


### What happened:

```
Test 1:  Tool returns full record with SSN, card, email
         → redact_pii finds PII → replaces with masked version
         → LLM receives: "SSN: ***-**-6789, Card: ****-****-****-9012, Email: [REDACTED]"
         → LLM responds to user with ONLY the redacted data

Test 2:  Tool returns record with no PII
         → redact_pii finds nothing → .allow() → LLM receives original data
```

The LLM **never sees** the raw PII. It can only share what the guardrail allows.

---

## Example 3: Both Input + Output Guards on One Tool

**Scenario: Banking Transfer Tool** — Validate arguments (no negative amounts, no self-transfer) AND redact account numbers from the confirmation.

In [5]:
import json
from agents import (
    Agent, Runner,
    function_tool, tool_input_guardrail, tool_output_guardrail,
    ToolGuardrailFunctionOutput,
)


# --- Input Guard: Validate transfer arguments ---
@tool_input_guardrail
def validate_transfer_args(data):
    """Business rule validation for transfers."""
    try:
        args = json.loads(data.context.tool_arguments or "{}")
    except json.JSONDecodeError:
        return ToolGuardrailFunctionOutput.reject_content("Invalid arguments format.")
    
    amount = args.get("amount", 0)
    from_acct = args.get("from_account", "")
    to_acct = args.get("to_account", "")
    
    # Rule 1: No negative or zero amounts
    if isinstance(amount, (int, float)) and amount <= 0:
        print(f"BLOCKED: Negative/zero amount ({amount})")
        return ToolGuardrailFunctionOutput.reject_content(
            f"Transfer amount must be positive. Got: {amount}"
        )
    
    # Rule 2: No self-transfers
    if from_acct and to_acct and from_acct == to_acct:
        print(f"BLOCKED: Self-transfer ({from_acct} → {to_acct})")
        return ToolGuardrailFunctionOutput.reject_content(
            "Cannot transfer to the same account."
        )
    
    # Rule 3: Max transfer limit
    if isinstance(amount, (int, float)) and amount > 1_000_000:
        print(f"BLOCKED: Over limit ({amount})")
        return ToolGuardrailFunctionOutput.reject_content(
            f"Transfer exceeds maximum limit of 1,000,000. Got: {amount}"
        )
    
    print(f"Transfer args valid: {amount} from {from_acct} to {to_acct}")
    return ToolGuardrailFunctionOutput.allow()


# --- Output Guard: Mask account numbers in confirmation ---
@tool_output_guardrail
def mask_accounts_in_output(data):
    """Replace full account numbers with masked versions in tool output."""
    import re
    text = str(data.output or "")
    # Mask any 8+ digit number
    masked = re.sub(r'\b(\d{4})(\d{4,})\b', lambda m: '****' + m.group(2)[-4:], text)
    if masked != text:
        print("Masked account numbers in output")
        return ToolGuardrailFunctionOutput.reject_content(masked)
    return ToolGuardrailFunctionOutput.allow()


# --- The guarded tool ---
@function_tool(
    tool_input_guardrails=[validate_transfer_args],
    tool_output_guardrails=[mask_accounts_in_output],
)
def transfer_money(from_account: str, to_account: str, amount: float) -> str:
    """Transfer money between bank accounts."""
    return (
        f"Transfer complete!\n"
        f"From: {from_account}\n"
        f"To: {to_account}\n"
        f"Amount: PKR {amount:,.0f}\n"
        f"Reference: TXN-{abs(hash(from_account)) % 100000:05d}"
    )


bank_agent = Agent(
    name="Bank Agent",
    instructions="You process bank transfers. Use the transfer_money tool. Be concise.",
    model=MODEL,
    tools=[transfer_money],
)


# Test 1: Valid transfer
print("--- Test 1: Valid Transfer ---")
result = await Runner.run(bank_agent, "Transfer 50000 from account 12345678 to 87654321")
print(f"Agent: {result.final_output[:250]}")

# Test 2: Negative amount
print("\n--- Test 2: Negative Amount ---")
result = await Runner.run(bank_agent, "Transfer -500 from 11111111 to 22222222")
print(f"Agent: {result.final_output[:200]}")

# Test 3: Over limit
print("\n--- Test 3: Over Limit ---")
result = await Runner.run(bank_agent, "Transfer 5000000 from 11111111 to 22222222")
print(f"Agent: {result.final_output[:200]}")

--- Test 1: Valid Transfer ---
Transfer args valid: 50000 from 12345678 to 87654321
Masked account numbers in output
Agent: Done. Transfer complete.

Reference: TXN-81011

--- Test 2: Negative Amount ---
BLOCKED: Negative/zero amount (-500)
Agent: The transfer amount must be positive. Please resend with a positive amount.

--- Test 3: Over Limit ---
BLOCKED: Over limit (5000000)
Agent: Transfer failed: amount exceeds the maximum limit of 1,000,000.


### What happened:

```
Test 1: Args valid → tool executes → output guardrail masks account numbers → LLM responds
Test 2: Input guard catches negative amount → .reject_content() → tool NEVER runs
        → LLM receives rejection message → tells user the amount is invalid
Test 3: Input guard catches over-limit → same rejection flow
```

---

## Example 4: SQL Injection Prevention on a Database Tool

**Scenario:** A tool takes natural language queries and runs them against a database. The LLM might be tricked into generating malicious SQL.

In [6]:
import json
from agents import (
    Agent, Runner,
    function_tool, tool_input_guardrail,
    ToolGuardrailFunctionOutput,
)


@tool_input_guardrail
def block_dangerous_sql(data):
    """Prevent destructive SQL in database query arguments."""
    args_str = (data.context.tool_arguments or "{}").upper()
    
    dangerous = [
        "DROP ", "DELETE ", "TRUNCATE ", "ALTER ", "UPDATE ",
        "INSERT ", "EXEC ", "EXECUTE ", "--", ";",
        "UNION SELECT", "OR 1=1", "OR '1'='1'",
    ]
    
    for pattern in dangerous:
        if pattern in args_str:
            print(f"SQL INJECTION BLOCKED: '{pattern.strip()}'")
            return ToolGuardrailFunctionOutput.reject_content(
                f"Dangerous SQL pattern detected: {pattern.strip()}. "
                f"Only SELECT queries are allowed."
            )
    
    print("SQL query looks safe")
    return ToolGuardrailFunctionOutput.allow()


@function_tool(
    tool_input_guardrails=[block_dangerous_sql],
)
def query_database(sql_query: str) -> str:
    """Execute a read-only SQL query against the database."""
    # Simulated results
    if "customer" in sql_query.lower():
        return "Results: [{name: 'Ahmed', plan: 'Enterprise'}, {name: 'Sara', plan: 'Pro'}]"
    return f"Query executed: {sql_query[:50]}... (0 rows)"


db_agent = Agent(
    name="DB Agent",
    instructions="You help users query the database. Convert their questions to SQL and use query_database. Only SELECT queries.",
    model=MODEL,
    tools=[query_database],
)


# Safe query
print("--- Test 1: Safe SELECT ---")
result = await Runner.run(db_agent, "Show me all enterprise customers")
print(f"Agent: {result.final_output[:200]}")

# SQL injection attempt
print("\n--- Test 2: SQL Injection ---")
result = await Runner.run(db_agent, "Show customers; DROP TABLE customers; --")
print(f"Agent: {result.final_output[:200]}")

--- Test 1: Safe SELECT ---
SQL INJECTION BLOCKED: ';'
SQL query looks safe
Agent: Here are the customers returned as enterprise-related:

- Ahmed — Enterprise
- Sara — Pro

If you want, I can also format this as a table or help narrow it down further.

--- Test 2: SQL Injection ---
Agent: I can’t execute destructive or non-query SQL.

If you want to view customers safely, I can run a `SELECT` مثل:

```sql
SELECT * FROM customers;
```

If you’d like, I can also help you add filters or l


---

## When to Use Which Guardrail

| Guardrail | Scope | Runs When | Best For |
|---|---|---|---|
| **Agent Input** | First agent only | Before/parallel with agent | Jailbreak, off-topic, input PII |
| **Agent Output** | Last agent only | After agent finishes | Response format, competitor mentions |
| **Tool Input** | Every tool call | Before tool executes | Argument validation, SQL injection, secrets |
| **Tool Output** | Every tool call | After tool executes | PII redaction, data masking, format validation |

In multi-agent systems with handoffs, **only tool guardrails run on every step**. Agent guardrails only run on the first/last agent. If you have tools that interact with databases, APIs, or sensitive data, tool guardrails are essential.

---

## ToolGuardrailFunctionOutput — The 3 Options

```python
# Option 1: ALLOW — let the tool run/return normally
return ToolGuardrailFunctionOutput.allow()

# Option 2: REJECT — block execution, send message back to LLM
#   For input guards: tool DOES NOT execute, LLM sees rejection message
#   For output guards: tool output is REPLACED with rejection message
return ToolGuardrailFunctionOutput.reject_content("Reason for rejection")

# Option 3: TRIPWIRE — halt the entire agent run with an exception
#   Use for severe violations where the conversation should stop
return ToolGuardrailFunctionOutput(tripwire_triggered=True)
```

| Method | Tool Runs? | Agent Continues? | Use Case |
|---|---|---|---|
| `.allow()` | Yes | Yes | Normal case |
| `.reject_content(msg)` | No (input) / output replaced (output) | Yes — LLM sees rejection | Recoverable violations |
| `tripwire_triggered=True` | No | No — exception raised | Critical violations |

---

## Important Limitations

Tool guardrails work ONLY on tools created with `@function_tool`. They do NOT work on:
- Handoff calls (use handoff `input_filter` instead)
- Hosted tools (WebSearchTool, FileSearchTool, CodeInterpreterTool)
- `Agent.as_tool()` calls
- MCP server tools

---

## Summary

| Decorator | Attaches To | Data Available | Returns |
|---|---|---|---|
| `@tool_input_guardrail` | `function_tool(tool_input_guardrails=[...])` | `data.context.tool_arguments` | `ToolGuardrailFunctionOutput` |
| `@tool_output_guardrail` | `function_tool(tool_output_guardrails=[...])` | `data.output` | `ToolGuardrailFunctionOutput` |

**Key takeaway:** Agent guardrails protect the edges (input/output). Tool guardrails protect the **internals** — every database query, every API call, every file write. In production, you need both.